In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Setup and Model Initialization

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [4]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

In [5]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [6]:
# State schema with ALL required LangGraph fields
from typing import TypedDict, List, Annotated, Optional
from langchain_core.messages import BaseMessage
import operator

class WeddingState(TypedDict):
    """Complete state schema for wedding coordinator"""
    messages: Annotated[List[BaseMessage], operator.add]
    remaining_steps: int
    origin: Optional[str]
    destination: Optional[str]
    guest_count: Optional[str]
    genre: Optional[str]
    wedding_date: Optional[str]

## Create Subagents

In [7]:
from langchain.agents import create_agent
from datetime import datetime

current_date = datetime.now().strftime("%Y-%m-%d")

travel_agent = create_agent(
    model=model,
    tools=tools,
    state_schema=WeddingState,
    system_prompt=f"""
    You are a travel agent. Today's date is {current_date}.
    
    Your task is to search for flights for a wedding on the SPECIFIC DATE provided.
    
    CRITICAL RULES:
    1. The 'departureDate' MUST be exactly the date provided in the request.
    2. If no date is provided, choose a reasonable date at least 2 months in the future.
    3. The date MUST be in YYYY-MM-DD format.
    4. NEVER search for a date before {current_date}.
    
    Return a concise summary of flight options with prices and times.
    """
)

In [8]:
venue_agent = create_agent(
    model=model,
    tools=[web_search],
    state_schema=WeddingState,
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location with the desired capacity.
    
    Find the best venue options based on:
    - Price (lowest first)
    - Capacity (must accommodate exact guest count)
    - Reviews (highest rated)
    
    Make multiple searches if needed. Return a structured summary with:
    Venue name, Location, Capacity, Price estimate, Review highlights.
    """
)

In [9]:
playlist_agent = create_agent(
    model=model,
    tools=[query_playlist_db],
    state_schema=WeddingState,
    system_prompt="""
    You are a playlist specialist. Query the SQL database to curate a wedding playlist for the given genre.
    
    Steps:
    1. Query the database for tracks matching the genre
    2. Calculate total duration and total cost (each song has a price)
    3. If queries fail, adjust and retry - do not give up
    4. Return: song titles, artists, total duration (minutes), total cost, brief rationale
    """
)

## Create Tools

In [10]:
from langchain_core.tools import tool, ToolRuntime
from langchain_core.messages import ToolMessage, HumanMessage
from langgraph.types import Command

@tool
async def search_flights(state: WeddingState) -> str:
    """Travel agent searches for flights using state values."""
    origin = (state.get("origin") or "").strip()
    destination = (state.get("destination") or "").strip()
    wedding_date = (state.get("wedding_date") or "").strip()
    
    if not all([origin, destination, wedding_date]):
        missing = [k for k, v in {"origin": origin, "destination": destination, "wedding_date": wedding_date}.items() if not v]
        return f"Error: Missing required fields: {', '.join(missing)}. Call update_state first."
    
    query = f"Find flights from {origin} to {destination} departing on {wedding_date} for a wedding. Return concise options with prices."
    
    agent_input = {k: v for k, v in state.items() if k not in ['messages', 'remaining_steps']}
    agent_input["messages"] = [HumanMessage(content=query)]
    
    response = await travel_agent.ainvoke(agent_input)
    
    messages = response.get("messages", [])
    if messages:
        last = messages[-1]
        if hasattr(last, "content"):
            return last.content
        return str(last)
    return "No flight results found."

In [11]:
@tool
def search_venues(state: WeddingState) -> str:
    """Venue agent finds the best venue for location and capacity."""
    destination = (state.get("destination") or "").strip()
    capacity = (state.get("guest_count") or "").strip()
    
    if not destination or not capacity:
        missing = [k for k, v in {"destination": destination, "guest_count": capacity}.items() if not v]
        return f"Error: Missing: {', '.join(missing)}. Call update_state first."
    
    query = f"Find wedding venues in {destination} for {capacity} guests. Prioritize low price, exact capacity, high reviews."
    
    agent_input = {k: v for k, v in state.items() if k not in ['messages', 'remaining_steps']}
    agent_input["messages"] = [HumanMessage(content=query)]
    
    response = venue_agent.invoke(agent_input)
    
    messages = response.get("messages", [])
    if messages:
        last = messages[-1]
        if hasattr(last, "content"):
            return last.content
        return str(last)
    return "No venue results found."

In [12]:
@tool
def suggest_playlist(state: WeddingState) -> str:
    """Playlist agent curates a wedding playlist for the given genre."""
    genre = (state.get("genre") or "").strip()
    
    if not genre:
        return "Error: Missing 'genre' in state. Call update_state first."
    
    query = f"Find {genre} tracks for wedding playlist. Include name, artist, duration, price. Calculate total duration and cost."
    
    agent_input = {k: v for k, v in state.items() if k not in ['messages', 'remaining_steps']}
    agent_input["messages"] = [HumanMessage(content=query)]
    
    response = playlist_agent.invoke(agent_input)
    
    messages = response.get("messages", [])
    if messages:
        last = messages[-1]
        if hasattr(last, "content"):
            return last.content
        return str(last)
    return "No playlist results found."

In [13]:
@tool
def update_state(
    origin: str, 
    destination: str, 
    guest_count: str, 
    genre: str, 
    wedding_date: str,
    runtime: ToolRuntime  # Required to access tool_call_id
) -> Command:
    """Update the WeddingState with all wedding details."""
    return Command(update={
        "origin": origin.strip() if origin else "",
        "destination": destination.strip() if destination else "",
        "guest_count": guest_count.strip() if guest_count else "",
        "genre": genre.strip() if genre else "",
        "wedding_date": wedding_date.strip() if wedding_date else "",
        # REQUIRED: ToolMessage with matching tool_call_id
        "messages": [ToolMessage(
            content="State updated successfully with wedding details.",
            tool_call_id=runtime.tool_call_id  # Matches the LLM's tool call
        )]
    })

## Create Coordinator

In [14]:
from langchain.agents import create_agent
from datetime import datetime

current_date_formatted = datetime.now().strftime("%A, %B %d, %Y")

coordinator = create_agent(
    model=model,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt=f"""
    You are a wedding coordinator. Today is {current_date_formatted}.
    
    TASK FLOW:
    1. Extract from user: origin, destination, guest_count, genre, wedding timing.
    2. Determine wedding_date:
       - Specific date provided → convert to YYYY-MM-DD
       - "in X months" → calculate from today ({current_date_formatted})
       - No date → default to 6 months from today
    3. IMMEDIATELY call update_state with ALL five values.
    4. AFTER state update, call specialists:
       - search_flights → flight options
       - search_venues → venue options
       - suggest_playlist → playlist with duration/cost
    5. Synthesize ALL outputs into ONE comprehensive summary.
    6. Format final response with sections: ✈️ Flights, 🏛️ Venue, 🎵 Playlist, 📅 Date, 💰 Costs.
    
    CRITICAL: Never call specialists before update_state. Wait for all results before summarizing.
    """
)

## Test

In [15]:
from langchain_core.messages import HumanMessage

initial_input = {
    "messages": [
        HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre, in about three months")
    ],
    "remaining_steps": 25,
    "origin": "",
    "destination": "",
    "guest_count": "",
    "genre": "",
    "wedding_date": ""
}

# Execute the coordinator agent
response = await coordinator.ainvoke(initial_input)

# Print the final consolidated wedding plan
print("\n" + "="*60)
print("🎊 COMPLETE WEDDING PLAN 🎊")
print("="*60)
print(response["messages"][-1].content)
print("="*60)

ValueError: Expected to have a matching ToolMessage in Command.update for tool 'update_state', got: []. Every tool call (LLM requesting to call a tool) in the message history MUST have a corresponding ToolMessage. You can fix it by modifying the tool to return `Command(update=[ToolMessage("Success", tool_call_id=tool_call_id), ...], ...)`.

link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r